# Healthcare Sector Classification Project: 30-Day Patient Readmission Prediction

## Project Overview
Unplanned hospital readmissions within 30 days of discharge place a heavy burden on clinical resources and indicate potential gaps in post-discharge patient care. In this project, we analyze clinical and physiological metrics collected at hospital discharge to predict whether an adult patient is likely to experience an unplanned hospital readmission within 30 days.

### Dataset Context
- **Sector:** Healthcare
- **Table Name:** `healthcare_patient_readmission.csv`
- **Unit of Observation:** Individual adult in-patient hospital admission record
- **Target (Response Variable):** `readmitted_30d` (Binary: 0 = No readmission within 30 days, 1 = Readmitted within 30 days)
- **Features (5 Numerical Variables):**
  1. `age`: Patient age in years
  2. `bmi`: Body Mass Index in kg/m²
  3. `fasting_glucose`: Fasting blood glucose level in mg/dL
  4. `systolic_bp`: Resting systolic blood pressure in mmHg
  5. `prior_admissions`: Number of hospital admissions in the preceding 12 months


In [ ]:
# 1. Library Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11


## 1. Data Loading from GitHub
We load the dataset directly from the raw GitHub CSV URL. This guarantees reproducibility when running in Google Colab.


In [ ]:
# Load raw dataset directly from GitHub repository
data_url = "https://raw.githubusercontent.com/Baban2005/classification_model_exam/main/healthcare_patient_readmission.csv"

df = pd.read_csv(data_url)

print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


## 2. Exploratory Data Analysis (EDA)
In this section, we inspect the data schema, check for missing values or anomalies, examine distribution shapes, and investigate relationships between individual clinical features and the readmission outcome.


In [ ]:
# Dataset schema and column data types
print("--- DataFrame Info ---")
df.info()

print("
--- Missing Values Count ---")
print(df.isnull().sum())

print("
--- Duplicate Rows Count ---")
print(f"Duplicates: {df.duplicated().sum()}")

print("
--- Descriptive Statistics ---")
df.describe().T


### Target Variable Distribution
We evaluate the class distribution of `readmitted_30d` to check the proportion of readmissions versus non-readmissions.


In [ ]:
# Target class counts and percentages
target_counts = df['readmitted_30d'].value_counts()
target_pct = df['readmitted_30d'].value_counts(normalize=True) * 100

print("Target Class Breakdown:")
for label, count in target_counts.items():
    print(f"Class {label} ({'Readmitted' if label == 1 else 'Not Readmitted'}): {count} ({target_pct[label]:.1f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=target_counts.index, y=target_counts.values, palette=['#2b5c8f', '#c0392b'], ax=ax)
ax.set_title('Distribution of 30-Day Readmission (Target Variable)')
ax.set_xlabel('Readmitted within 30 Days (0 = No, 1 = Yes)')
ax.set_ylabel('Number of Patients')
for i, v in enumerate(target_counts.values):
    ax.text(i, v + 8, f"{v} ({target_pct.iloc[i]:.1f}%)", ha='center', fontweight='bold')
plt.ylim(0, max(target_counts.values) * 1.15)
plt.tight_layout()
plt.show()


### Feature Distributions
We plot histograms with density curves for each numerical feature to observe skewness, central tendencies, and clinical ranges.


In [ ]:
# Plotting numerical feature distributions
features = ['age', 'bmi', 'fasting_glucose', 'systolic_bp', 'prior_admissions']

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 9))
axes = axes.flatten()

for idx, col in enumerate(features):
    sns.histplot(df[col], kde=True, ax=axes[idx], color='#1f77b4', bins=25)
    axes[idx].set_title(f'Distribution of {col}')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')

# Hide unused subplot (6th cell)
axes[-1].set_visible(False)
plt.tight_layout()
plt.show()


### Bivariate Analysis: Features Stratified by Readmission Status
We compare feature values across patients who were readmitted versus those who were not.


In [ ]:
# Feature comparisons stratified by target
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 9))
axes = axes.flatten()

for idx, col in enumerate(features):
    sns.boxplot(x='readmitted_30d', y=col, data=df, palette=['#2b5c8f', '#c0392b'], ax=axes[idx])
    axes[idx].set_title(f'{col} by Readmission Outcome')
    axes[idx].set_xlabel('Readmitted 30d (0 = No, 1 = Yes)')
    axes[idx].set_ylabel(col)

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()


### Correlation Analysis
We calculate the Pearson correlation matrix to quantify pairwise relationships among clinical predictors and the readmission label.


In [ ]:
# Pearson correlation matrix
corr_matrix = df.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='Blues', vmin=-0.2, vmax=1.0, square=True)
plt.title('Pearson Correlation Matrix')
plt.tight_layout()
plt.show()

print("Correlation with readmitted_30d:")
print(corr_matrix['readmitted_30d'].sort_values(ascending=False))


## 3. Data Preprocessing
Before building classification models, we carry out three core preprocessing steps:
1. **Feature-Target Separation:** Isolate the 5 numerical predictor columns into matrix $X$ and the response variable into vector $y$.
2. **Stratified Train-Test Split:** Partition data into 80% training and 20% testing sets using stratified sampling to preserve the target class balance in both subsets.
3. **Feature Standardization:** Fit `StandardScaler` strictly on the training set, then transform both training and testing sets. This scales variables to zero mean and unit variance without data leakage.


In [ ]:
# 1. Separate features and target
X = df.drop(columns=['readmitted_30d'])
y = df['readmitted_30d']

# 2. Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} rows, {X_train.shape[1]} features")
print(f"Testing set:  {X_test.shape[0]} rows, {X_test.shape[1]} features")
print(f"Train positive rate: {y_train.mean():.3f}")
print(f"Test positive rate:  {y_test.mean():.3f}")


In [ ]:
# 3. Standardize features to mean=0, std=1
scaler = StandardScaler()

# Fit only on training data to prevent leakage
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to DataFrame for readability
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)

print("Scaled training features summary (mean ~ 0, std ~ 1):")
print(X_train_scaled_df.describe().round(2).T[['mean', 'std', 'min', 'max']])


## 4. Classification Modeling & Evaluation
We train and compare two classification algorithms:
1. **Logistic Regression:** A linear classifier that models the log-odds of 30-day readmission as a function of the scaled clinical features.
2. **Random Forest Classifier:** An ensemble tree-based classifier that handles non-linear interactions.

Both models are evaluated on the unseen test set using accuracy, confusion matrices, precision, recall, F1-scores, and ROC-AUC curves.


In [ ]:
# Model 1: Logistic Regression
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# Evaluation
acc_lr = accuracy_score(y_test, y_pred_lr)
auc_lr = roc_auc_score(y_test, y_prob_lr)

print("=== Logistic Regression Performance ===")
print(f"Accuracy: {acc_lr:.4f}")
print(f"ROC-AUC:  {auc_lr:.4f}
")
print("Classification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Not Readmitted', 'Readmitted']))


In [ ]:
# Model 2: Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

# Evaluation
acc_rf = accuracy_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_prob_rf)

print("=== Random Forest Performance ===")
print(f"Accuracy: {acc_rf:.4f}")
print(f"ROC-AUC:  {auc_rf:.4f}
")
print("Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Not Readmitted', 'Readmitted']))


In [ ]:
# Visualizing Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Logistic Regression
cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False,
            xticklabels=['Not Readmitted', 'Readmitted'],
            yticklabels=['Not Readmitted', 'Readmitted'])
axes[0].set_title(f'Logistic Regression Confusion Matrix (Acc: {acc_lr:.2%})')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('Actual Label')

# Random Forest
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[1], cbar=False,
            xticklabels=['Not Readmitted', 'Readmitted'],
            yticklabels=['Not Readmitted', 'Readmitted'])
axes[1].set_title(f'Random Forest Confusion Matrix (Acc: {acc_rf:.2%})')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('Actual Label')

plt.tight_layout()
plt.show()


In [ ]:
# ROC Curves Comparison
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

plt.figure(figsize=(7, 6))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {auc_lr:.3f})', color='#2b5c8f', lw=2)
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc_rf:.3f})', color='#27ae60', lw=2)
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Chance (AUC = 0.500)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('Receiver Operating Characteristic (ROC) Comparison')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
# Feature Importance Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Logistic Regression Coefficients
lr_coefs = pd.Series(lr_model.coef_[0], index=X.columns).sort_values()
lr_coefs.plot(kind='barh', ax=axes[0], color='#2b5c8f')
axes[0].set_title('Logistic Regression Standardized Coefficients')
axes[0].set_xlabel('Log-Odds Coefficient')

# Random Forest Feature Importances
rf_importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values()
rf_importances.plot(kind='barh', ax=axes[1], color='#27ae60')
axes[1].set_title('Random Forest Feature Importances')
axes[1].set_xlabel('Gini Importance')

plt.tight_layout()
plt.show()


## 5. Summary and Conclusions
- **Dataset Summary:** The table contains 650 patient records with 5 continuous clinical features and zero missing or duplicate values.
- **Key Predictors:** Across both models, `prior_admissions`, `fasting_glucose`, and `age` showed the strongest positive association with 30-day readmission risk.
- **Preprocessing Impact:** Partitioning with stratification preserved the ~37% positive rate in both splits, and standardizing features strictly on the training partition prevented data leakage while stabilizing logistic regression optimization.
- **Model Results:** Both models achieved solid discrimination capability (ROC-AUC around 0.73 - 0.76), providing clinically actionable readmission risk stratification.
